In [13]:
import torch
from pathlib import Path
import torchaudio
from torch.utils.data import Dataset
import pandas as pd
import soundfile as sf

In [2]:
print (torch.cuda.is_available())
print(torch.cuda.get_device_properties())
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024 ** 3):.2f} GB')
print(f'CUDA Version: {torch.version.cuda}')

True
_CudaDeviceProperties(name='Quadro M1200', major=5, minor=0, total_memory=4095MB, multi_processor_count=5, uuid=4d8c5a0b-9b85-fbdc-50ea-1a8d85b2f0d3, L2_cache_size=2MB)
VRAM: 4.00 GB
CUDA Version: 12.4


In [3]:
files = Path('../data/train_audio').rglob('*.ogg')
print(f'number of files in train_audio folder: {len(list(files))}')
import pandas as pd
train_df = pd.read_csv('../data/train.csv')
print(f'number of data rows in train.csv: {len(train_df)}')



number of files in train_audio folder: 35549
number of data rows in train.csv: 35549


In [4]:

train_df.head(10)

,primary_label,secondary_labels,type,latitude,longitude,scientific_name,common_name,class_name,inat_taxon_id,author,license,rating,url,filename,collection
0,1161364,[],[],-22.7562,-46.8666,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1216197....,1161364/iNat1216197.ogg,iNat
1,1161364,[],[],-22.7558,-46.8700,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1114648....,1161364/iNat1114648.ogg,iNat
2,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/810195.m...,1161364/iNat810195.ogg,iNat
3,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/818781.m...,1161364/iNat818781.ogg,iNat
4,1161364,[],[],-22.7426,-46.8985,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/556514.m...,1161364/iNat556514.ogg,iNat
5,1161364,[],[],-22.0843,-47.7327,Guyalna cuta,Guyalna cuta,Insecta,1161364,Carlos Otávio Gussoni,cc-by-nc,0.0,https://static.inaturalist.org/sounds/868369.w...,1161364/iNat868369.ogg,iNat
6,1161364,[],[],-20.9848,-43.7609,Guyalna cuta,Guyalna cuta,Insecta,1161364,Pedro Cavalcante,cc-by-nc,0.0,https://static.inaturalist.org/sounds/842139.w...,1161364/iNat842139.ogg,iNat
7,1161364,[],[],-19.9674,-43.9895,Guyalna cuta,Guyalna cuta,Insecta,1161364,Pedro Cavalcante,cc-by-nc,0.0,https://static.inaturalist.org/sounds/840159.m...,1161364/iNat840159.ogg,iNat
8,1161364,[],[],-19.8713,-43.9607,Guyalna cuta,Guyalna cuta,Insecta,1161364,Alexandre S. Michelotto,cc0,0.0,https://static.inaturalist.org/sounds/1264238....,1161364/iNat1264238.ogg,iNat
9,1161364,[],[],-18.8358,-40.7354,Guyalna cuta,Guyalna cuta,Insecta,1161364,Vitor C. Dias Gonçalves,cc-by-nc,0.0,https://static.inaturalist.org/sounds/869958.m...,1161364/iNat869958.ogg,iNat


### Check that files in subfolders are the same as the entries in train.csv


In [5]:
folders = [f for f in Path('../data/train_audio').iterdir() if f.is_dir()]
print(f'number of folders in train_audio: {len(folders)}')
for folder in folders:
    files_in_folder = [str(f.relative_to('../data/train_audio/')) for f in folder.rglob('*.ogg')]
    sub_df = train_df[train_df['primary_label']== folder.name]['filename']
    files_set = set(files_in_folder)
    sub_df_set = set(sub_df)
    missing_in_folder = sub_df_set - files_set
    missing_in_csv = files_set - sub_df_set
    if (len(missing_in_csv)!=0) or (len(missing_in_csv)!=0): 
        print(f'{folder}: In df but not in folder: ', missing_in_folder)
        print(f'{folder}: In folder but not in df: ', missing_in_csv)

number of folders in train_audio: 206


### Chunking the long files


In [6]:
def chunk_long_file(file_path, output_dir, chunk_duration=5):
    # Load the audio file
    waveform, sample_rate = torchaudio.load(file_path)
    
    # Calculate the number of samples for the chunk duration
    chunk_samples = int(chunk_duration * sample_rate)
    
    # Get the total number of samples in the waveform
    total_samples = waveform.size(1)
    
    # Calculate the number of chunks
    num_chunks = total_samples // chunk_samples
    
    # Create output directory if it doesn't exist
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Split and save chunks
    for i in range(num_chunks):
        start_sample = i * chunk_samples
        end_sample = (i + 1) * chunk_samples
        chunk = waveform[:, start_sample:end_sample]
        
        # Save the chunk
        output_file = output_dir / f"{file_path.stem}_chunk_{i+1}.wav"
        torchaudio.save(output_file, chunk, sample_rate)
        print(f"Saved {output_file}")


In [11]:
reduced_df = train_df[['primary_label']].copy()
reduced_df['file_path'] = train_df.apply(lambda row: f"../data/train_audio/{row['filename']}", axis=1)
reduced_df['offset_sec'] = 0
reduced_df.head(30)

,primary_label,file_path,offset_sec
0,1161364,../data/train_audio/1161364/iNat1216197.ogg,0
1,1161364,../data/train_audio/1161364/iNat1114648.ogg,0
2,1161364,../data/train_audio/1161364/iNat810195.ogg,0
3,1161364,../data/train_audio/1161364/iNat818781.ogg,0
4,1161364,../data/train_audio/1161364/iNat556514.ogg,0
5,1161364,../data/train_audio/1161364/iNat868369.ogg,0
6,1161364,../data/train_audio/1161364/iNat842139.ogg,0
7,1161364,../data/train_audio/1161364/iNat840159.ogg,0
8,1161364,../data/train_audio/1161364/iNat1264238.ogg,0
9,1161364,../data/train_audio/1161364/iNat869958.ogg,0


In [10]:
print(reduced_df.iloc[0, 2])

0


In [27]:
FIXED_LENGTH = 5 # seconds
inserted_df = pd.DataFrame(columns = ['primary_label', 'file_path', 'offset_sec'])
for i in range(len(train_df)):
    file_path = '../data/train_audio/'+ train_df['filename'][i]
    primary_label = train_df['primary_label'][i]
    info = sf.info(file_path)
    if (info.duration % FIXED_LENGTH != 0):
        for j in range(int(info.duration/FIXED_LENGTH)+1):
            new_row = pd.DataFrame([{'primary_label': primary_label, 
                       'file_path': file_path, 
                       'offset_sec': j*FIXED_LENGTH}])
            inserted_df = pd.concat([inserted_df, new_row], ignore_index=True)
    else:
        for j in range(int(info.duration/FIXED_LENGTH)):
            new_row = pd.DataFrame([{'primary_label': primary_label, 
                       'file_path': file_path, 
                       'offset_sec': j*FIXED_LENGTH}])
            inserted_df = pd.concat([inserted_df, new_row], ignore_index=True)
inserted_df.head(30)

,primary_label,file_path,offset_sec
0,1161364,../data/train_audio/1161364/iNat1216197.ogg,0
1,1161364,../data/train_audio/1161364/iNat1216197.ogg,5
2,1161364,../data/train_audio/1161364/iNat1216197.ogg,10
3,1161364,../data/train_audio/1161364/iNat1216197.ogg,15
4,1161364,../data/train_audio/1161364/iNat1114648.ogg,0
5,1161364,../data/train_audio/1161364/iNat1114648.ogg,5
6,1161364,../data/train_audio/1161364/iNat1114648.ogg,10
7,1161364,../data/train_audio/1161364/iNat1114648.ogg,15
8,1161364,../data/train_audio/1161364/iNat1114648.ogg,20
9,1161364,../data/train_audio/1161364/iNat1114648.ogg,25


In [28]:
inserted_df.to_csv('../inserted_df.csv', index=False)

In [ ]:
import math
import soundfile as sf

FIXED_LENGTH = 5

# Add duration and num_segments columns vectorized
train_df['file_path'] = '../data/train_audio/' + train_df['filename']
train_df['duration'] = train_df['file_path'].apply(lambda p: sf.info(p).duration)
train_df['num_segments'] = (train_df['duration'] / FIXED_LENGTH).apply(math.ceil)

# Repeat each row num_segments times
inserted_df = train_df.loc[train_df.index.repeat(train_df['num_segments'])][['primary_label', 'file_path']].copy()

# Assign offset_sec
inserted_df['offset_sec'] = inserted_df.groupby(level=0).cumcount() * FIXED_LENGTH
inserted_df = inserted_df.reset_index(drop=True)


In [ ]:


i = 0
info = sf.info(reduced_df.iloc[i,1])
print(f'Samplerate: {info.samplerate} Hz.\n frames: {info.frames}\n channesl: {info.channels}\n duration: {info.duration}\n format:{info.format}\n subtype: {info.subtype}')

a = int((info.duration/FIX_LENGTH)) # -64 to remove the pad of silence sapmle added by OGG Vorbis when compressing audio blocks.
b = info.duration % FIX_LENGTH
print(a)
print(b)
if b != 0:
    for j in range(a):
        new_row = pd.DataFrame({'primary_label':reduced_df['primary_label'][i], 'filepath': reduced_df['file_path'][i],\
                               'offset_sec': j+1})
        reduced_df_copy = pd.concat
        

Samplerate: 32000 Hz.
 frames: 576768
 channesl: 1
 duration: 18.024
 format:OGG
 subtype: VORBIS
3
3.024000000000001


In [ ]:
class BirdSoundDataset(Dataset):
    def __init__(self, df, audio_dir, transform=None):
        self.df = df
        self.audio_dir = Path(audio_dir)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file = self.audio_dir / row['filename']
        waveform, sample_rate = torchaudio.load(str(file))
        if self.transform:
            waveform = self.transform(waveform)
        label = row['label_encoded']
        return waveform, label
    
dataset = BirdDataset(train_df, '../data/train_audio')


6


55
